## Лабораторная работа #6. Решение ОДУ

Импорт библиотек и модулей

In [ ]:
import numpy as np
from comp_math.ode.ode_registry import ODERegistry
from comp_math.linear_algebra.objects.matrix import Matrix
from comp_math.linear_algebra.objects.vector import Vector
import matplotlib.pyplot as plt

Условия задачи VIII.11.4

In [ ]:
def f(tn: float, yn: Vector):
    x = yn[0]
    y = yn[1]
    z = yn[2]
    u = yn[3]
    f = Vector([0, 0, 0, 0])
    f[0] = z # x'
    f[1] = u # y'
    f[2] = - x / (x ** 2 + y ** 2) ** (3 / 2)
    f[3] = - y / (x ** 2 + y ** 2) ** (3 / 2)
    return f

y0 = [0.5, 0, 0, np.sqrt(3)]
t_span = [0, 2]
h = 0.01
dim = 4

In [ ]:
methods = [
    (ODERegistry.create_solver("euler"), "Метод Эйлера"),
    (ODERegistry.create_solver("heun"),  "Метод Хойна"),
    (ODERegistry.create_solver("kutta"), "Метод Кутта 3"),
    (ODERegistry.create_solver("rk4"), "Метод RK4"),
    (ODERegistry.create_solver("gauss_legendre_2"), "Метод Гаусса-Лежандра"),
    (ODERegistry.create_solver("rado"), "Метод Радо"),
]

Функции для построения графиков

In [ ]:
def make_subplot(axes, t, y, ind, label):
    axes[ind].plot(t, list(map(lambda yn: yn[ind], y)), marker='o')
    axes[ind].set_title(label + "(t)")
    axes[ind].set_xlabel("time")
    axes[ind].set_ylabel(label)
    axes[ind].grid(True, alpha=0.3)
    # axes[ind].set_xticks(rotation=45)


def make_plot(t, y, title=""): 
    fig, axes = plt.subplots(4, 1, figsize=[8,12])

    make_subplot(axes, t, y, 0, "x")
    make_subplot(axes, t, y, 1, "y")
    make_subplot(axes, t, y, 2, "z")
    make_subplot(axes, t, y, 3, "u")

    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


In [ ]:
for method in methods:
    t, y = method[0].solve(f, dim, t_span, y0, h)
    make_plot(t, y, method[1])

In [ ]:
from matplotlib.animation import FuncAnimation
from mpl_toolkits.mplot3d import Axes3D

t, y = methods[0][0].solve(f, dim, t_span, y0, h)

t = t[::500]
y = y[::500]

fig = plt.figure(figsize=[10, 8])
ax = fig.add_subplot(111, projection='3d')

line, = ax.plot(list(map(lambda yn: yn[0], y)), 
                list(map(lambda yn: yn[1], y)),
                list(map(lambda yn: yn[2], y)), 
                'b-', alpha=0.5, label='Траектория')

point, = ax.plot([y[0][0]], [y[0][1]], [y[0][2]], 
                 'ro', markersize=8, label='Точка')

ax.set_xlim(-1.5, 1.5)
ax.set_ylim(-1.5, 1.5)
ax.set_zlim(0, 8)
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
ax.set_title('Движение точки по 3D траектории')
ax.legend()

def update(frame):
    point.set_data([y[frame][0]], [y[frame][1]])
    point.set_3d_properties([y[frame][2]])
    
    # Изменение угла обзора
    # ax.view_init(elev=30, azim=frame/2)
    
    return point,

anim = FuncAnimation(fig, update, frames=len(t), interval=50, blit=False)

plt.show() 

# Сохранить анимацию 
anim.save('3d_point_animation.mp4', writer='ffmpeg', fps=30)